# FIFA 2026 Data Repository — Exploratory Analysis

Interactive notebook for exploring the FIFA World Cup 2022 dataset (base for 2026 modeling).

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from database import get_connection
from reports import *

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Database Overview

In [ ]:
from database import table_counts
table_counts()

## 2. Top Scorers

In [ ]:
scorers = report_top_scorers(20)
scorers.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=scorers.head(15), x='goals', y='player_name', palette='viridis', ax=ax)
ax.set_title('Top 15 Goal Scorers — FIFA World Cup 2022')
ax.set_xlabel('Goals')
ax.set_ylabel('Player')
plt.tight_layout()
plt.show()

## 3. xG vs Goals (Finishing Efficiency)

In [ ]:
with get_connection() as conn:
    df = pd.read_sql_query("""
        SELECT player_name, team_name, position, goals, xg, minutes_played
        FROM players
        WHERE minutes_played > 90 AND (goals > 0 OR xg > 1)
        ORDER BY xg DESC
    """, conn)

df['goals_minus_xg'] = df['goals'] - df['xg']

fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=df, x='xg', y='goals', hue='team_name', size='minutes_played',
                sizes=(50, 400), alpha=0.8, ax=ax)
ax.plot([0, df['xg'].max()], [0, df['xg'].max()], 'k--', alpha=0.5, label='xG = Goals')
ax.set_title('Goals vs Expected Goals (xG) — FIFA 2022')
ax.set_xlabel('xG')
ax.set_ylabel('Actual Goals')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Team Summary

In [ ]:
teams = report_team_summary()
teams.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=teams, x='goals_for', y='goals_against', hue='team_name', s=200, ax=ax)
ax.set_title('Goals For vs Goals Against by Team')
ax.set_xlabel('Goals For')
ax.set_ylabel('Goals Against')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. Venue Analysis for 2026

In [ ]:
with get_connection() as conn:
    venues = pd.read_sql_query("""
        SELECT * FROM venues_2026 ORDER BY altitude_m DESC
    """, conn)

venues[['venue_name', 'city', 'altitude_m', 'avg_june_temp_c', 'avg_humidity_pct', 'climate_zone']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=venues, x='altitude_m', y='venue_name', ax=axes[0], palette='coolwarm')
axes[0].set_title('Venue Altitude (m)')

sns.barplot(data=venues, x='avg_june_temp_c', y='venue_name', ax=axes[1], palette='YlOrRd')
axes[1].set_title('Average June Temperature (°C)')

plt.tight_layout()
plt.show()

## 6. Player Search

In [ ]:
search_players('Mbappé')

## 7. Custom SQL Queries

In [ ]:
with get_connection() as conn:
    # Most cards
    cards = pd.read_sql_query("""
        SELECT player_name, team_name, yellow_cards, red_cards,
               (yellow_cards + red_cards) as total_cards,
               cards_per_90
        FROM players
        WHERE minutes_played > 0
        ORDER BY total_cards DESC
        LIMIT 15
    """, conn)

cards